# XGBoost: A Hands-On Lesson

XGBoost (eXtreme Gradient Boosting) is one of the most widely used libraries for tabular prediction. It builds gradient-boosted trees, the same family of models as scikit-learn's `GradientBoostingRegressor`, but it's faster and adds features that matter for insurance work:

- **Objectives** for counts (`count:poisson`), positive skewed amounts (`reg:gamma`), and Tweedie data (`reg:tweedie`)
- **Monotonic constraints**, which force a feature's effect to move in one direction
- **Early stopping**, which picks the number of trees automatically
- **Sample weights**, which let you model rates using exposure

This lesson covers:

1. Setup and simulated data
2. Train/test split
3. A first regression model
4. Early stopping
5. Feature importance
6. Poisson objective for claim frequency (with exposure as weights)
7. Gamma objective for claim severity
8. Monotonic constraints
9. The native API (`DMatrix` and `xgb.train`)
10. Saving and loading models
11. Practice exercises, with solutions

The data is simulated, so nothing needs downloading. Run the cells in order.

## 1. Setup and simulated data

In [ ]:
import os

# Avoids a hang with OpenMP threading on some systems (including WSL2)
os.environ["OMP_NUM_THREADS"] = "1"

import numpy as np
import pandas as pd
import xgboost as xgb

print("xgboost", xgb.__version__)

We simulate policies with a few features and a claim count per policy. Each policy has an exposure (the fraction of the year it was in force), and claims depend on age, smoking, and urban location.

In [ ]:
rng = np.random.default_rng(seed=3)
n = 20000

policies = pd.DataFrame({
    "age": rng.integers(18, 80, size=n),
    "bmi": rng.normal(27, 4.5, size=n).clip(15, 50).round(1),
    "smoker": rng.choice([0, 1], size=n, p=[0.8, 0.2]),
    "urban": rng.choice([0, 1], size=n, p=[0.6, 0.4]),
    "exposure": rng.uniform(0.25, 1.0, size=n),
})

true_rate = np.exp(
    -2.4
    + 0.012 * (policies["age"] - 40)
    + 0.4 * policies["smoker"]
    + 0.25 * policies["urban"]
    + 0.01 * (policies["bmi"] - 27)
)
policies["claims"] = rng.poisson(true_rate * policies["exposure"])

# Claim severity: average cost per claim, with a skewed (gamma) spread
severity_mean = np.exp(8.0 + 0.004 * (policies["age"] - 40) + 0.3 * policies["smoker"])

policies.head()

In [ ]:
print("policies:", len(policies))
print("total claims:", policies["claims"].sum())
print("claims per policy-year:", round(policies["claims"].sum() / policies["exposure"].sum(), 3))

## 2. Train/test split

Split by row here, because each row is one policy. Always hold out data the model never sees while training. We keep the test set for the final check.

In [ ]:
from sklearn.model_selection import train_test_split

features = ["age", "bmi", "smoker", "urban"]

train, test = train_test_split(policies, test_size=0.25, random_state=3)

X_train, y_train = train[features], train["claims"]
X_test, y_test = test[features], test["claims"]

print("train rows:", len(train), " test rows:", len(test))

Notice that the feature table uses `[features]`, a list of column names. XGBoost accepts pandas DataFrames directly.

## 3. A first regression model

XGBoost's scikit-learn wrapper, `XGBRegressor`, follows the same `fit` and `predict` pattern you already know. The first model below uses the default squared-error objective, just to see the API. We'll switch to a better objective for claim counts shortly.

In [ ]:
from sklearn.metrics import mean_absolute_error

model = xgb.XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    n_jobs=1,
    random_state=3,
)
model.fit(X_train, y_train)

pred = model.predict(X_test)
print("test MAE (claim counts):", round(mean_absolute_error(y_test, pred), 4))

Key settings:

- **`n_estimators`**: the number of trees.
- **`learning_rate`**: how much each tree contributes. Smaller values need more trees.
- **`max_depth`**: how complex each tree is.
- **`n_jobs`**: how many threads to use. We set `1` to stay safe on WSL2.

## 4. Early stopping

Too many trees overfit, and too few underfit. **Early stopping** watches a validation set and stops adding trees once the score stops improving. It's better than guessing `n_estimators`.

We split the training data again to make a validation set. The test set stays untouched.

In [ ]:
X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=3)

model = xgb.XGBRegressor(
    n_estimators=2000,            # an upper limit, not the final count
    learning_rate=0.05,
    max_depth=4,
    early_stopping_rounds=50,
    n_jobs=1,
    random_state=3,
)
model.fit(X_fit, y_fit, eval_set=[(X_val, y_val)], verbose=False)

print("best iteration:", model.best_iteration)
print("test MAE:", round(mean_absolute_error(y_test, model.predict(X_test)), 4))

Training stopped at the best iteration, so the model no longer has a fixed size set by hand.

## 5. Feature importance

XGBoost reports how much each feature contributed to the trees. A quick view:

In [ ]:
importance = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)
importance.round(3)

Feature importance shows *how much the model uses* a feature, not whether its effect is causal or even direction-aware. For a direction-aware view, check the partial dependence later, or use SHAP values. Treat importance as a starting point.

## 6. Poisson objective for claim frequency

Counts aren't normally distributed, so the squared-error default is a poor fit. The **Poisson objective** (`count:poisson`) fits counts properly.

Policies have different exposures, so we model the **rate** (claims per policy-year). Then we pass exposure as **sample weights**. A policy in force for half a year counts half as much, so the model learns from the rate and the exposure is handled correctly.

In [ ]:
rate = policies["claims"] / policies["exposure"]

train_rate = rate.loc[train.index]
train_exposure = train["exposure"]

freq_model = xgb.XGBRegressor(
    objective="count:poisson",
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=4,
    early_stopping_rounds=50,
    n_jobs=1,
    random_state=3,
)

X_fit, X_val, rate_fit, rate_val, exp_fit, exp_val = train_test_split(
    X_train, train_rate, train_exposure, test_size=0.2, random_state=3
)

freq_model.fit(
    X_fit, rate_fit,
    sample_weight=exp_fit,
    eval_set=[(X_val, rate_val)],
    sample_weight_eval_set=[exp_val],
    verbose=False,
)

print("best iteration:", freq_model.best_iteration)

In [ ]:
test_rate = rate.loc[test.index]
test_exposure = test["exposure"]

pred_rate = freq_model.predict(X_test)

# Compare expected and actual claim counts on the test set (exposure-weighted)
expected_claims = (pred_rate * test_exposure).sum()
actual_claims = test["claims"].sum()
print(f"expected claims: {expected_claims:,.1f}")
print(f"actual claims:   {actual_claims:,}")
print(f"A/E ratio:       {actual_claims / expected_claims:.3f}")

An A/E ratio near 1.0 means the model predicts the total number of claims correctly. This is the same calibration check used in the GLM lesson.

## 7. Gamma objective for claim severity

Claim costs are positive and right-skewed, which fits the **Gamma objective** (`reg:gamma`). Here we model the cost of each claim, using only policies that had claims.

In [ ]:
# Build one row per claim, with that policy's features
claim_rows = []
for _, row in policies[policies["claims"] > 0].iterrows():
    for _ in range(int(row["claims"])):
        claim_rows.append(row)

claims = pd.DataFrame(claim_rows).reset_index(drop=True)

In [ ]:
# Simulate each claim's cost from its own features (gamma spread around the mean)
claims["cost"] = rng.gamma(
    shape=2.0,
    scale=np.exp(8.0 + 0.004 * (claims["age"] - 40) + 0.3 * claims["smoker"]) / 2.0,
)

sev_train, sev_test = train_test_split(claims, test_size=0.25, random_state=3)

sev_model = xgb.XGBRegressor(
    objective="reg:gamma",
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=3,
    early_stopping_rounds=50,
    n_jobs=1,
    random_state=3,
)
sev_model.fit(
    sev_train[features], sev_train["cost"],
    eval_set=[(sev_test[features], sev_test["cost"])],
    verbose=False,
)

print("best iteration:", sev_model.best_iteration)
print("mean actual cost (test):   ", round(sev_test["cost"].mean(), 0))
print("mean predicted cost (test):", round(sev_model.predict(sev_test[features]).mean(), 0))

Note that the check uses the **mean**. With a Gamma objective, predictions estimate the average cost, so the averages should line up.

The claim table is built with `iterrows`, which is slow on large data. For real projects, use a vectorized method such as `Series.repeat`. For a demonstration this size, the loop is fine.

## 8. Monotonic constraints

Actuaries often need an effect to move in a known direction. For example, a higher BMI should never *reduce* predicted claims. Monotonic constraints enforce that during training:

- `1` means the prediction must not decrease as the feature increases.
- `-1` means it must not increase.
- `0` means no constraint.

In [ ]:
# Constrain: age (0), bmi (1), smoker (2), urban (3). Only BMI is constrained here,
# to be non-decreasing.
constrained = xgb.XGBRegressor(
    objective="count:poisson",
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=4,
    monotone_constraints="(0,1,0,0)",
    early_stopping_rounds=50,
    n_jobs=1,
    random_state=3,
)
constrained.fit(
    X_fit, rate_fit,
    sample_weight=exp_fit,
    eval_set=[(X_val, rate_val)],
    sample_weight_eval_set=[exp_val],
    verbose=False,
)

# Check: predictions should never decrease as BMI rises, holding other features fixed
probe = X_test.iloc[[0]].copy()
bmi_grid = np.linspace(15, 50, 15)
probe_rows = pd.concat([probe] * len(bmi_grid), ignore_index=True)
probe_rows["bmi"] = bmi_grid
curve = constrained.predict(probe_rows)

print("non-decreasing in BMI:", bool(np.all(np.diff(curve) >= -1e-9)))

Monotonic constraints are a good example of why actuaries like boosting. You get flexible fitting, but you can still enforce the direction an expert expects.

## 9. The native API

The scikit-learn wrapper is convenient, but XGBoost's own API is what most of its documentation uses. It works with a `DMatrix` data structure and `xgb.train`. The concepts are the same, and the native API gives finer control over training:

In [ ]:
dtrain = xgb.DMatrix(X_fit, label=rate_fit, weight=exp_fit)
dval = xgb.DMatrix(X_val, label=rate_val, weight=exp_val)

params = {
    "objective": "count:poisson",
    "eta": 0.05,
    "max_depth": 4,
    "nthread": 1,
    "seed": 3,
}

booster = xgb.train(
    params,
    dtrain,
    num_boost_round=2000,
    evals=[(dval, "val")],
    early_stopping_rounds=50,
    verbose_eval=False,
)

print("best iteration:", booster.best_iteration)

`eta` is the native name for `learning_rate`, and `nthread` is the native name for `n_jobs`. The native `Booster` object can also save and load models in XGBoost's own format.

## 10. Saving and loading

Save the model in XGBoost's own format. This avoids the security risks of pickling, and it keeps the model readable by other XGBoost versions in the same family:

In [ ]:
import tempfile
from pathlib import Path

model_path = Path(tempfile.gettempdir()) / "claims_frequency.json"
freq_model.save_model(model_path)

reloaded = xgb.XGBRegressor()
reloaded.load_model(model_path)

print("predictions match:", np.allclose(reloaded.predict(X_test), freq_model.predict(X_test)))

JSON is human-readable, which helps when reviewing a model. Use `.ubj` for a smaller binary format.

## 11. Practice exercises

Try each exercise before you look at the solution. Cells marked `# TODO` are for you to fill in.

**Exercise 1.** Change `max_depth` in the frequency model to 2 and to 6. Which gives a lower MAE on the test rates? Use `freq_model`'s objective and the test rates.

In [ ]:
# TODO

<details><summary>Solution</summary>

```python
from sklearn.metrics import mean_absolute_error

test_rate = rate.loc[test.index]
for depth in [2, 4, 6]:
    m = xgb.XGBRegressor(
        objective="count:poisson", n_estimators=2000, learning_rate=0.05,
        max_depth=depth, early_stopping_rounds=50, n_jobs=1, random_state=3,
    )
    m.fit(X_fit, rate_fit, sample_weight=exp_fit,
          eval_set=[(X_val, rate_val)], sample_weight_eval_set=[exp_val], verbose=False)
    print(depth, round(mean_absolute_error(test_rate, m.predict(X_test)), 4))
```

Very deep trees can overfit, and very shallow ones can miss interactions. The best depth depends on the data.

</details>

**Exercise 2.** Add a monotonic constraint that forces predictions to *not increase* with age (`-1` for age). Does it change the test A/E ratio much?

In [ ]:
# TODO

<details><summary>Solution</summary>

```python
m = xgb.XGBRegressor(
    objective="count:poisson", n_estimators=2000, learning_rate=0.05,
    max_depth=4, monotone_constraints="(-1,0,0,0)",
    early_stopping_rounds=50, n_jobs=1, random_state=3,
)
m.fit(X_fit, rate_fit, sample_weight=exp_fit,
      eval_set=[(X_val, rate_val)], sample_weight_eval_set=[exp_val], verbose=False)
```

Here the constraint is wrong for this data, since claims rise with age in the simulation. A constraint that contradicts the data usually hurts accuracy. Constraints should come from domain knowledge, not from guesses.

</details>

**Exercise 3.** Compare the predicted severity for a smoker and a non-smoker of the same age and BMI. Which is higher, and by how much?

In [ ]:
# TODO

<details><summary>Solution</summary>

```python
base = X_test.iloc[[0]].copy()
base["smoker"] = 0
smoker = base.copy()
smoker["smoker"] = 1

print("non-smoker:", round(sev_model.predict(base)[0], 0))
print("smoker:    ", round(sev_model.predict(smoker)[0], 0))
```

The smoker prediction should be higher, since the simulation builds that effect in. Compare the ratio to `exp(0.3)`, which is about 1.35.

</details>

## Summary

- **`XGBRegressor`** follows the familiar `fit`/`predict` pattern.
- **Early stopping** sets the number of trees from validation data.
- **`count:poisson`** with **exposure as weights** models claim frequency.
- **`reg:gamma`** models positive, skewed costs such as severity.
- **Monotonic constraints** enforce expert knowledge about direction.
- **The native API** (`DMatrix`, `xgb.train`) gives finer control.
- **Save models in XGBoost's format**, not pickle.